In [1]:
import numpy as np
import math

In [2]:
X = np.arange(200).reshape(100, 2)
y = np.array([0] * 90 + [1] * 10)
rng = np.random.default_rng(42)

In [3]:
def train_test_split(X, y, test_size=0.25, stratify=None, shuffle=True, rng=None):
    if rng is None:                      
        rng = np.random.default_rng()

    X = np.asarray(X)
    y = np.asarray(y)
    n = len(y)

    if stratify is not None:               
        stratify = np.asarray(stratify)
        idx_treino, idx_teste = [], []
        for cls in np.unique(stratify):                
            idx_cls = np.where(stratify == cls)[0]     
            idx_cls = rng.permutation(idx_cls)
            corte = math.ceil(round(len(idx_cls) * test_size, 10))
            idx_treino.extend(idx_cls[:len(idx_cls) - corte])
            idx_teste.extend(idx_cls[len(idx_cls) - corte:])

        idx_treino = rng.permutation(idx_treino)
        idx_teste = rng.permutation(idx_teste)
    else:
        indices = rng.permutation(n) if shuffle else np.arange(n)
        corte = math.ceil(round(n * test_size, 10))
        idx_treino = indices[:n - corte]
        idx_teste = indices[n - corte:]

    return X[idx_treino], X[idx_teste], y[idx_treino], y[idx_teste]

In [4]:
x_treino, x_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, stratify=y,rng=rng)

In [5]:
class KFold:
    def __init__(self, n_splits = 5, shuffle = False, rng = None):
        self.n_splits = n_splits
        self.rng = rng
        self.shuffle = shuffle
        if rng is None:                      
                self.rng = np.random.default_rng()
        if n_splits < 2:
            print("Numero de splits menor que o minimo setando para o padrao")
            self.n_splits = 5 
        

    def get_n_splits(self):
        return self.n_splits

    def split(self, X, y):
        X = np.asarray(X)
        n = len(X)
        tamanhos = np.full(self.n_splits, n // self.n_splits)
        tamanhos[: n % self.n_splits] += 1

        indices = self.rng.permutation(n) if self.shuffle else np.arange(n)

        inicio = 0

        for tamanho in tamanhos:
             fim = inicio + tamanho
             teste = indices[inicio:fim]
             treino = np.concatenate((indices[:inicio], indices[fim:]))
             yield treino, teste
             inicio = fim




    

In [6]:
kf = KFold(n_splits=3)
y = np.arange(10)
for treino, teste in kf.split(np.arange(10), y):
    print("treino:", treino, " teste:", teste)

treino: [4 5 6 7 8 9]  teste: [0 1 2 3]
treino: [0 1 2 3 7 8 9]  teste: [4 5 6]
treino: [0 1 2 3 4 5 6]  teste: [7 8 9]


In [7]:
def mean_squared_error(y_true,y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    return np.mean((y_true - y_pred) ** 2)

In [8]:
y_true = [10, 20, 30]
y_pred = [12, 18, 27]
loss = mean_squared_error(y_true, y_pred)
print(loss)

5.666666666666667


In [9]:
def root_mean_squared_error(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

In [10]:
y_true = [10, 20, 30]
y_pred = [11, 21, 35]
loss = root_mean_squared_error(y_true, y_pred)
print(loss)

3.0


In [11]:
def mean_absolute_error(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.mean(np.abs(y_true - y_pred))

In [12]:
y_true = [10, 20, 30]
y_pred = [12, 15, 33]
loss = mean_absolute_error(y_true, y_pred)
print(loss)

3.3333333333333335


In [13]:
def r2_score(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)

    if ss_tot == 0:
        return 1.0 if ss_res == 0 else 0.0

    return 1 - (ss_res / ss_tot)

In [14]:
y_true = [3, -0.5, 2]
y_pred = [2.5, 0.0, 2]
loss = r2_score(y_true, y_pred)
print(loss)

0.9230769230769231


In [15]:
y_true = np.array([0, 0, 0, 1, 1, 1, 1, 0])
y_pred = np.array([0, 1, 0, 1, 0, 1, 1, 0])
k = 2

In [17]:
codigo = k * y_true + y_pred
codigo

array([0, 1, 0, 3, 2, 3, 3, 0])

In [18]:
contagem = np.bincount(codigo, minlength=k * k)
print(contagem)   # [3 1 1 3]

[3 1 1 3]


In [ ]:
def confusion_matrix(y_true, y_pred):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    n = len(y_true)

    todos = np.concatenate((y_true, y_pred))
    classes, posicoes = np.unique(todos, return_inverse=True)
    k = len(classes)

    y_true = posicoes[:n]
    y_pred = posicoes[n:]

    matriz = np.bincount(k * y_true + y_pred, minlength= k * k).reshape(k,k)
    return matriz

